#Fase 2: Generación de datos textuales sintéticos

## Librerías

In [ ]:
!pip install -U llama-cpp-python

## Carga del modelo

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    n_ctx= 2048 #tamaño del contexto: tokens de entrada y token generados
)

# se opta por una cifra intermedia ya que cuanto mayor sea, más memoria se consume
# porque el modelo tiene que guardar más “estado interno” por cada token del prompt.

## Generación de texto

### **A) Técnicas de prompting directas**

* Zero-shot
* One-shot
* Few-shot
* Topic-controlled prompting
* Randomized prompting
* Zero-shot topic generation
* Instruction based prompting

#### **Elementos esenciales**

Estas 7 técnicas, tienen como base de funcionamiento los mismos elementos esenciales, aunque no todas los usen todos. Estos elementos son:
* Instrucción base
* Contexto
* Demostraciones
* Restricciones globales

**1. Intrucción base**

Es la misma para todas las técnicas ➡ *"Escribe una reseña textual realista, similar a la que redactaría un cliente real, sobre un producto de un pequeño negocio local."*

**2. Contexto**

Son las condiciones bajo las cuales se genera cada reseña. Tenemos un conjunto base de condiciones cerrado (producto, perfil de cliente, valoración y tono) y el tema de la reseña. El tema será implícito o explícito según la técnica.

**3. Demostraciones**

Son ejemplos reales de reseñas utilizados como guía en algunas técnicas. Estas no corresponden necesariamente al mismo producto o valoración que las condiciones de generación, ya que su función es transmitir patrones de estilo y estructura propios de reseñas reales, y no contenido específico.

Las demostraciones han sido extraidas de un dataset de reseñas de amazon de Kaggle ([texto del enlace](https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews)). Se han seleccionado 5 reseñas de ejemplo de tonos variados y longitud similar a la que se pedirá al LLM. Originalmente estaban en inglés pero han sido traducidas al español.

**4. Restricciones goblales**

Son reglas formales que se aplican a todas las técnicas de forma implícita. Incluyen el número de palabras, el tipo de salida, el formato y exclusiones como no mencionar la palabra IA. Aseguran uniformaidad entre los resultados para poder compararlos entre sí.


#### **Técnicas básicas**

**1. Zero-shot**

Esta técnica se usará como base ya que permite ver como se comporta el modelo con solo la tarea y el contexto mínimo. Sirve para medir la calidad base (qué tan realista es sin "ayuda"), la diversidad espontánea (si se repite el mismo patrón de prompt, ¿tiende a repetirse?) y el "sesgo natural" del modelo, es decir, en qué aspectos se fija sin nuestras indicaciones.

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.  
* Restricciones globales.

**2. One-shot**

Con esta técnica vemos la capacidad de aprendizaje en contexto con un solo ejemplo. Sirve para observar si el modelo imita el formato y el estilo del ejemplo dado, si mejora la coherencia y realismo respecto al zero-shot y hasta qué punto sacrifica diversidad al anclarse a un único patrón.

En comparación con zero-shot suele generar textos más “correctos” y consistentes, pero existe el riesgo de que todas las reseñas se parezcan demasiado entre sí.

¿Mejora la calidad si muestro un único ejemplo representativo, sin llegar a sobrecondicionar al modelo?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.
* Demostraciones: una.
* Restricciones globales.


**3. Few-shot**

Con esta técnica vemos la capacidad del modelo para aprender patrones más estables cuando se le proporcionan varios ejemplos. Permite observar si aumenta la fidelidad (estructura, estilo, nivel de detalle), si las reseñas resultan menos genéricas y si aparece una pérdida de diversidad por repetición de patrones.

En comparación con one-shot suele mejorar la calidad media, pero incrementa el riesgo de outputs muy parecidos entre sí y dependencia excesiva del estilo de los ejemplos.

¿Hasta qué punto puedo mejorar la calidad del texto guiando al modelo con ejemplos, y a qué coste en diversidad?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.  
* Demostraciones: varias.
* Restricciones globales.


Estas tres técnicas representan los enfoques básicos para la generación de texto con LLMs, diferenciándose principalmente por el nivel de información y ejemplos proporcionados al modelo. Estas estrategias permiten analizar el equilibrio entre diversidad y fidelidad en los datos sintéticos generados y constituyen el punto de partida para técnicas más avanzadas que introducen un mayor control sobre atributos específicos del texto.

#### **Técnicas avanzadas**

**4. Topic-controlled prompting**

Con esta técnica pasamos a mirar la capacidad del modelo para seguir una condición semántica explícita impuesta desde el prompt. Permite observar si el modelo centra el contenido de la reseña en el tema indicado, si mantiene coherencia temática a lo largo del texto y si es capaz de integrar el tema sin entrar en contradicción con el producto, el perfil del cliente, la valoración y el tono.

En comparación con zero-shot, one-shot y few-shot el foco del contenido ya no es decidido libremente por el modelo, reduciendo la variabilidad temática, pero aumenta el control sobre qué se está evaluando.

¿Hasta qué punto el modelo es capaz de generar reseñas coherentes cuando se le impone explícitamente el foco del contenido?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: explícito.  
* Restricciones globales.


**5. Randomized prompting**

Ahora ya no se evalúa únicamente la capacidad del modelo para cumplir una condición concreta, sino su capacidad para generar un conjunto diverso de reseñas manteniendo una calidad media estable. En concreto, se analiza el efecto de introducir variabilidad controlada en el proceso de generación y cómo esta variabilidad influye en la diversidad global del dataset, sin comprometer la coherencia interna ni el realismo de los textos generados.

A diferencia de topic-controlled prompting, el tema no se fija manualmente para cada generación, sino que se selecciona de forma aleatoria a partir de un conjunto predefinido, lo que permite aumentar la cobertura temática sin intervención manual directa.

¿Hasta qué punto es posible generar de forma automática un conjunto amplio y diverso de reseñas coherentes y realistas, manteniendo una calidad media aceptable bajo variación temática controlada?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: aleatorio dentro de un conjunto definido previamente.  
* Restricciones globales.


**6. Zero-shot topic generation**

Con esta técnica el modelo genera temas relevantes a partir del contexto dado para cada generación. Sirve para evaluar la capacidad del modelo para identificar temas pertinentes para una reseña, si los temas generados son coherentes con el producto, el perfil del cliente, la valoración y el tono y si el uso de temas auto-generados permite aumentar la diversidad del contenido sin necesidad de definir previamente un espacio temático cerrado.

A diferencia de topic-controlled y randomized prompting el tema no es impuesto por el diseñador del prompt, ni se selecciona de forma aleatoria a partir de una lista, sino que emerge directamente del propio modelo en un paso previo a la generación de la reseña.

¿Hasta qué punto un LLM es capaz de proponer temas relevantes y generar reseñas coherentes a partir de ellos sin intervención humana directa en la selección temática?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: generado.  
* Restricciones globales.

**7.  Instruction based prompting**

Con esta técnica se evalúa principalmente la capacidad del modelo para seguir instrucciones explícitas y estructuradas, más allá de condiciones aisladas como el tema. Permite observar si el modelo interpreta correctamente una instrucción descompuesta en partes claras (tarea, contexto, condiciones y formato), si mejora la alineación con los requisitos formales del texto, y si una formulación más estructurada del prompt reduce ambigüedades y resultados no deseados.

A diferencia de las técnicas anteriores no se introduce diversidad mediante ejemplos (one/few-shot), ni mediante variación o generación de temas (randomized o zero-shot topic generation), sino que el control se ejerce a través de la claridad y explicitud de la instrucción.

¿Hasta qué punto una formulación explícita y estructurada de la instrucción mejora la alineación del modelo con los requisitos de la tarea?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: explícito.  
* Restricciones globales.

#### Configuración común

In [ ]:
import os
import re
import json
import random
from datetime import datetime
from typing import Dict, Any, Optional, List, Tuple

import pandas as pd
from tqdm import tqdm

In [ ]:
# Configuración general
MIN_WORDS = 90
MAX_WORDS = 130

# Dataset viable computacionalmente
N_CONDITIONS = 20
K_PER_CONDITION = 5  # nº de muestras por condición y técnica

# Parámetros de generación
GEN_DEFAULT = dict(
    max_tokens=220, #equivalente al número de palabras en tokens
    temperature=0.8,
    top_p=0.9,
)

# Carpeta de salida
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Contexto

# Condiciones base

PRODUCT_CONDITIONS = [
    {"product": "Cafetera italiana",        "persona": "persona que prepara café en casa a diario",          "rating": 5, "tone": "cercano"},
    {"product": "Sartén antiadherente",     "persona": "cocinero aficionado",                                "rating": 4, "tone": "neutral"},
    {"product": "Batidora de mano",         "persona": "persona que hace smoothies en casa",                 "rating": 3, "tone": "informal"},
    {"product": "Tostadora",                "persona": "persona con desayunos rápidos",                      "rating": 2, "tone": "crítico"},
    {"product": "Juego de tuppers",         "persona": "persona que lleva comida al trabajo",                "rating": 4, "tone": "cercano"},
    {"product": "Robot aspirador básico",   "persona": "hogar con mascota",                                  "rating": 2, "tone": "crítico"},
    {"product": "Escoba de microfibra",     "persona": "persona que limpia su piso semanalmente",            "rating": 4, "tone": "neutral"},
    {"product": "Organizador de armario",   "persona": "persona que busca orden en casa",                    "rating": 3, "tone": "formal"},
    {"product": "Lámpara de escritorio LED","persona": "estudiante universitario",                           "rating": 4, "tone": "neutral"},
    {"product": "Soporte para portátil",    "persona": "teletrabajador",                                     "rating": 5, "tone": "formal"},
    {"product": "Ratón ergonómico",         "persona": "persona que pasa muchas horas frente al ordenador",  "rating": 3, "tone": "neutral"},
    {"product": "Cuaderno",                 "persona": "estudiante que toma apuntes a mano",                 "rating": 4, "tone": "informal"},
    {"product": "Botella",                  "persona": "persona que va a la oficina",                       "rating": 5, "tone": "cercano"},
    {"product": "Mochila urbana",           "persona": "viajero de fin de semana",                           "rating": 4, "tone": "neutral"},
    {"product": "Power bank portátil",      "persona": "persona que viaja con frecuencia",                  "rating": 2, "tone": "crítico"},
    {"product": "Secador de pelo",          "persona": "persona que se arregla por las mañanas",            "rating": 4, "tone": "neutral"},
    {"product": "Crema hidratante",         "persona": "persona con piel sensible",                         "rating": 5, "tone": "cercano"},
    {"product": "Cepillo de pelo",          "persona": "persona que cuida su cabello a diario",             "rating": 3, "tone": "formal"},
    {"product": "Altavoz Bluetooth",        "persona": "persona que escucha música en casa",                "rating": 4, "tone": "cercano"},
    {"product": "Manta para sofá",          "persona": "persona que busca confort en casa",                 "rating": 5, "tone": "cercano"},
]

# Banco de temas

REVIEW_TOPICS = [
    "experiencia de uso",
    "durabilidad",
    "calidad",
    "relación calidad-precio",
    "cumplimiento de expectativas",
    "diseño y aspecto general del producto",
    "comodidad",
    "funcionalidad frente a lo prometido",
    "nivel de satisfacción general",
    "aspectos que podrían mejorarse",
    "tamano y espacio",
    "sensación de durabilidad",
    "comparación con productos similares",
    "adecuación al estilo de vida del usuario",
    "atención al cliente",
    "experiencia de compra",
    "primeras impresiones frente a experiencia real",
    "envío",
    "grado de recomendación a otras personas",
    "valor aportado en el contexto personal"
]

# Demostraciones:

# ONE-SHOT

DEMO_ONE_SHOT_ES = """
En primer lugar, me gustó el formato y el tono del libro (la forma en que la autora se dirige al lector).
Sin embargo, no sentí que aportara ninguno de los secretos internos que el libro prometía revelar.
Si estás empezando a informarte sobre la facultad de derecho y no conoces todos los requisitos de admisión,
entonces este libro puede ser de gran ayuda. Si ya has hecho tus deberes y estás buscando una ventaja adicional en el proceso de admisión,
recomiendo libros más específicos por tema. Por ejemplo, libros sobre cómo escribir tu declaración personal,
libros centrados específicamente en la preparación del LSAT (los libros de Powerscore fueron los más útiles para mí),
y también hay algunas páginas web con muy buenos consejos dirigidos a ayudar a las personas a las que vas a pedir cartas de recomendación.
Aun así, para quienes son nuevos en todo este proceso, este libro puede aclarar perfectamente los requisitos.
""".strip()

# FEW-SHOT: usamos solo 2 y de longitud más reducida para no superar el máximo de tokens que admite el modelo en el prompt (521)

DEMOS_FEW_SHOT_ES = [
"""Mi querida Pat tiene una de las GRANDES voces de su generación. He escuchado este CD durante AÑOS y todavía ME ENCANTA.
Cuando estoy de buen humor me hace sentir aún mejor. Un mal humor simplemente se evapora como azúcar bajo la lluvia. Este CD rebosa VIDA.
Las voces son simplemente IMPRESIONANTES y las letras son demoledoras. Una de las joyas ocultas de la vida. Para mí, este es un CD de isla
desierta. Por qué nunca llegó a ser famosa está más allá de mi comprensión. Cada vez que lo pongo, da igual si son negros, blancos, jóvenes,
mayores, hombres o mujeres, TODO EL MUNDO dice lo mismo: “¿Quién estaba cantando?”
""".strip(),
"""
Compré este cargador en julio de 2003 y funcionó bien durante un tiempo. El diseño es bonito y cómodo. Sin embargo, después de aproximadamente un
año, las baterías ya no mantenían la carga. Casi que mejor comprar pilas alcalinas desechables, o buscar otro cargador que venga con baterías que
duren más.
""".strip()

]

In [ ]:
# funciones comunes


from google.colab import files
import os
import pandas as pd


def constraints_block() -> str:
    return f"""Restricciones globales:
- Longitud: {MIN_WORDS}-{MAX_WORDS} palabras.
- Contenido: Incluye detalles concretos sobre la experiencia de uso.
- Estilo: natural y humano, propio de un cliente real.
- Formato: un único párrafo, sin listas ni numeración.
- Prohibido: mencionar modelos de lenguaje, IA o sistemas automáticos.
- Salida: devuelve únicamente el texto final de la reseña.
"""

def base_instruction() -> str:
    return "Escribe una reseña textual realista, similar a la que redactaría un cliente real, sobre un producto de un negocio local."

def instance_block(p: Dict[str, Any]) -> str:
    return f"""Condiciones:
Producto: {p["product"]}
Perfil del cliente: {p["persona"]}
Valoración: {p["rating"]}/5
Tono: {p["tone"]}
"""

def condition_id(p: Dict[str, Any]) -> str:
    return f'{p["product"]}|{p["persona"]}|{p["rating"]}|{p["tone"]}'

def chat_generate(prompt: str, gen_params: Dict[str, Any]) -> str:
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        **gen_params
    )
    text = out["choices"][0]["message"]["content"].strip()
    text = re.sub(r"\n{2,}", "\n", text).strip()
    return text

def make_row(sample_id: int,
             technique: str,
             p: Dict[str, Any],
             prompt: str,
             text: str,
             topic: Optional[str] = None,
             attempts: Optional[int] = None) -> Dict[str, Any]:
    return {
        "sample_id": sample_id,
        "technique": technique,
        "condition_id": condition_id(p),
        "product": p["product"],
        "persona": p["persona"],
        "rating": p["rating"],
        "tone": p["tone"],
        "topic": topic,
        "prompt": prompt,
        "text": text,
        "attempts": attempts,
    }

def save_outputs(df: pd.DataFrame, filename_base: str) -> None:
    csv_path = os.path.join(OUT_DIR, f"{filename_base}.csv")
    xlsx_path = os.path.join(OUT_DIR, f"{filename_base}.xlsx")

    df.to_csv(csv_path, index=False, encoding="utf-8")
    df.to_excel(xlsx_path, index=False)

    print("Guardado:")
    print("-", csv_path)
    print("-", xlsx_path)

    files.download(csv_path)
    files.download(xlsx_path)

def quick_summary(df: pd.DataFrame, name: str) -> None:
    print(f"\n{name} | n={len(df)}")

#### Generación...

##### Zero-shot

In [ ]:
def build_prompt_zero_shot(p: Dict[str, Any]) -> str:
    """
    Zero-shot:
    - Instrucción base
    - Contexto (producto + persona + rating + tono)
    - Restricciones globales
    - Tema implícito (no se menciona)
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_zero_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ZERO-SHOT"):
        prompt = build_prompt_zero_shot(p)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="zero_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_zero_shot = generate_zero_shot_dataset()

# Resumen y guardado
quick_summary(df_zero_shot, "ZERO-SHOT")
save_outputs(df_zero_shot, "zero_shot")
df_zero_shot.head()


##### One-shot

In [ ]:
def build_prompt_one_shot(p: Dict[str, Any], demo: str = DEMO_ONE_SHOT_ES) -> str:
    """
    One-shot:
    - Instrucción base
    - Contexto (producto + persona + rating + tono)
    - 1 demostración (para estilo/estructura)
    - Restricciones globales
    - Tema implícito (no se menciona)
    """
    return "\n".join([
        base_instruction(),
        "",
        "Ejemplo (para guiar estilo y estructura):",
        demo.strip(),
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_one_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    demo: str = DEMO_ONE_SHOT_ES,
) -> pd.DataFrame:
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ONE-SHOT"):
        prompt = build_prompt_one_shot(p, demo=demo)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="one_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_one_shot = generate_one_shot_dataset()

# Resumen y guardado
quick_summary(df_one_shot, "ONE-SHOT")
save_outputs(df_one_shot, "one_shot")
df_one_shot.head()

##### Few-shot

In [ ]:
def build_prompt_few_shot(p: Dict[str, Any], demos: List[str] = DEMOS_FEW_SHOT_ES) -> str:
    """
    Few-shot:
    - Instrucción base
    - Contexto (producto + persona + rating + tono)
    - Varias demostraciones (para estilo/estructura)
    - Restricciones globales
    - Tema implícito (no se menciona)
    """
    demos_block = "\n\n".join([d.strip() for d in demos if d and d.strip()])

    return "\n".join([
        base_instruction(),
        "",
        "Ejemplos (para guiar estilo y estructura):",
        demos_block,
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_few_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    demos: List[str] = DEMOS_FEW_SHOT_ES,
) -> pd.DataFrame:
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando FEW-SHOT"):
        prompt = build_prompt_few_shot(p, demos=demos)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="few_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_few_shot = generate_few_shot_dataset()

# Resumen y guardado
quick_summary(df_few_shot, "FEW-SHOT")
save_outputs(df_few_shot, "few_shot")
df_few_shot.head()

##### Topic-controlled

In [ ]:
def build_prompt_topic_controlled(p: Dict[str, Any], topic: str) -> str:
    """
    Topic-controlled prompting:
    - Instrucción base
    - Contexto (producto + persona + rating + tono)
    - Tema explícito (asignado manualmente)
    - Restricciones globales
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (explícito): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_topic_controlled_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    """
    Asignación manual 1:1 en orden:
    - conditions[i] usa topics[i]
    Requisito: len(conditions) == len(topics)
    """
    if len(conditions) != len(topics):
        raise ValueError(
            f"Topic-controlled requiere len(conditions) == len(topics). "
            f"Ahora: {len(conditions)} condiciones vs {len(topics)} temas."
        )

    rows = []
    sample_id = 0

    for i, p in enumerate(tqdm(conditions, desc="Generando TOPIC-CONTROLLED")):
        topic = topics[i]
        prompt = build_prompt_topic_controlled(p, topic=topic)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="topic_controlled",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_topic_controlled = generate_topic_controlled_dataset()

# Resumen y guardado
quick_summary(df_topic_controlled, "TOPIC-CONTROLLED")
save_outputs(df_topic_controlled, "topic_controlled")
df_topic_controlled.head()


##### Randomized prompting


In [ ]:
def build_prompt_randomized(p: Dict[str, Any], topic: str) -> str:
    """
    Randomized prompting:
    - Instrucción base
    - Contexto (producto + persona + rating + tono)
    - Tema explícito (seleccionado aleatoriamente)
    - Restricciones globales
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (aleatorio): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_randomized_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Para cada muestra (no solo por condición), el tema se elige aleatoriamente
    del banco REVIEW_TOPICS para aumentar la diversidad temática.
    """
    rng = random.Random(seed)

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando RANDOMIZED"):
        for _ in range(k_per_condition):
            topic = rng.choice(topics)
            prompt = build_prompt_randomized(p, topic=topic)

            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="randomized",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_randomized = generate_randomized_dataset()

# Resumen y guardado
quick_summary(df_randomized, "RANDOMIZED")
save_outputs(df_randomized, "randomized")
df_randomized.head()

##### Zero-shot topic generation

In [ ]:
def build_prompt_topic_proposal(p: Dict[str, Any]) -> str:
    """
    Paso 1: generar un tema relevante y coherente con:
    producto + persona + rating + tono.
    Salida estricta: SOLO el tema (frase corta), sin explicaciones.
    """
    return "\n".join([
        "Genera UN único tema breve y relevante para una reseña, coherente con las siguientes condiciones.",
        "Devuelve SOLO el tema (una frase corta), sin comillas, sin viñetas, sin explicación.",
        "",
        instance_block(p).strip(),
    ]).strip()


def clean_topic(raw: str) -> str:
    """
    Limpieza mínima para quedarnos con un tema utilizable:
    - primera línea
    - sin comillas envolventes
    - sin puntos finales largos
    """
    t = raw.strip().splitlines()[0].strip()
    t = t.strip('"\'')

    # elimina prefijos típicos (por si el modelo no obedece al 100%)
    t = re.sub(r"^(tema|tópico)\s*:\s*", "", t, flags=re.IGNORECASE).strip()

    # recorta espacios múltiples
    t = re.sub(r"\s{2,}", " ", t).strip()

    return t


def build_prompt_zero_shot_topic_generation(p: Dict[str, Any], topic: str) -> str:
    """
    Paso 2: generar la reseña usando el tema auto-generado.
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (generado): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_zero_shot_topic_generation_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    topic_gen_params: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    """
    Para cada reseña:
    1) Genera un tema condicionado (prompt corto)
    2) Genera la reseña con ese tema (prompt completo)
    """
    if topic_gen_params is None:
        topic_gen_params = dict(gen_params)
        topic_gen_params.update(dict(max_tokens=40, temperature=0.4, top_p=0.9))

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ZERO-SHOT TOPIC GENERATION"):
        for _ in range(k_per_condition):
            # Paso 1: proponer tema
            prompt_topic = build_prompt_topic_proposal(p)
            raw_topic = chat_generate(prompt_topic, topic_gen_params)
            topic = clean_topic(raw_topic)

            # Paso 2: reseña con tema generado
            prompt_review = build_prompt_zero_shot_topic_generation(p, topic=topic)
            text = chat_generate(prompt_review, gen_params)

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="zero_shot_topic_generation",
                    p=p,
                    prompt=prompt_review,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_zero_shot_topic_generation = generate_zero_shot_topic_generation_dataset()

# Resumen y guardado
quick_summary(df_zero_shot_topic_generation, "ZERO-SHOT TOPIC GENERATION")
save_outputs(df_zero_shot_topic_generation, "zero_shot_topic_generation")
df_zero_shot_topic_generation.head()

In [ ]:
# Ahora vamos a probar con one-shot en la generación de temas porque los que ha generado no eran como esperaba

DEMO_TOPIC_ONE_SHOT = "relación calidad-precio"  # ejemplo guía

def build_prompt_topic_proposal_one_shot(
    p: Dict[str, Any],
    demo_topic: str = DEMO_TOPIC_ONE_SHOT
) -> str:
    """
    Paso 1 (one-shot): generar un tema relevante y coherente con:
    producto + persona + rating + tono.

    Incluye 1 demostración de tema para guiar el tipo de salida.
    Salida estricta: solo el tema (frase corta), sin explicaciones.
    """
    return "\n".join([
        "Genera un único tema breve y relevante para una reseña, coherente con las siguientes condiciones.",
        "Devuelve solo el tema (una frase corta), sin comillas, sin viñetas, sin explicación.",
        "",
        "Ejemplo de tema:",
        demo_topic.strip(),
        "",
        instance_block(p).strip(),
    ]).strip()


def clean_topic(raw: str) -> str:
    """
    Limpieza mínima para quedarnos con un tema utilizable:
    - primera línea
    - sin comillas envolventes
    - sin puntos finales largos
    """
    t = raw.strip().splitlines()[0].strip()
    t = t.strip('"\'')

    # elimina prefijos típicos (por si el modelo no obedece al 100%)
    t = re.sub(r"^(tema|tópico)\s*:\s*", "", t, flags=re.IGNORECASE).strip()

    # recorta espacios múltiples
    t = re.sub(r"\s{2,}", " ", t).strip()

    return t


def build_prompt_zero_shot_topic_generation(p: Dict[str, Any], topic: str) -> str:
    """
    Paso 2: generar la reseña usando el tema auto-generado.
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema: {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_one_shot_topic_generation_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    topic_gen_params: Optional[Dict[str, Any]] = None,
    demo_topic: str = DEMO_TOPIC_ONE_SHOT,
) -> pd.DataFrame:
    """
    Para cada reseña:
    1) Genera un tema (ONE-SHOT con un tema ejemplo)
    2) Genera la reseña con ese tema (prompt completo)
    """
    if topic_gen_params is None:
        # más determinista para que el tema salga limpio y estable
        topic_gen_params = dict(gen_params)
        topic_gen_params.update(dict(max_tokens=40, temperature=0.4, top_p=0.9))

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ONE-SHOT TOPIC GENERATION"):
        for _ in range(k_per_condition):
            # Paso 1: proponer tema (one-shot)
            prompt_topic = build_prompt_topic_proposal_one_shot(p, demo_topic=demo_topic)
            raw_topic = chat_generate(prompt_topic, topic_gen_params)
            topic = clean_topic(raw_topic)

            # Paso 2: reseña con tema generado
            prompt_review = build_prompt_zero_shot_topic_generation(p, topic=topic)
            text = chat_generate(prompt_review, gen_params)

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="one_shot_topic_generation",
                    p=p,
                    prompt=prompt_review,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_one_shot_topic_generation = generate_one_shot_topic_generation_dataset()

# Resumen y guardado
quick_summary(df_one_shot_topic_generation, "ONE-SHOT TOPIC GENERATION")
save_outputs(df_one_shot_topic_generation, "one_shot_topic_generation")
df_one_shot_topic_generation.head()

no mejora mucho la verdad

##### Instruction based prompting


In [ ]:
def build_prompt_instruction_based(p: Dict[str, Any], topic: str) -> str:
    """
    Instruction-based prompting:
    - Control mediante una instrucción explícita y estructurada
    - Usa tema explícito (aquí lo asignamos 1:1 en orden, como en topic-controlled)
    - Sin demostraciones
    """
    return f"""TAREA
{base_instruction()}

CONTEXTO
{instance_block(p).strip()}

FOCO (TEMA)
{topic}

REQUISITOS
{constraints_block().strip()}
""".strip()


def generate_instruction_based_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    """
    Asignación 1:1 en orden:
    - conditions[i] usa topics[i]
    Requisito: len(conditions) == len(topics)
    """
    if len(conditions) != len(topics):
        raise ValueError(
            f"Instruction-based requiere len(conditions) == len(topics). "
            f"Ahora: {len(conditions)} condiciones vs {len(topics)} temas."
        )

    rows = []
    sample_id = 0

    for i, p in enumerate(tqdm(conditions, desc="Generando INSTRUCTION-BASED")):
        topic = topics[i]
        prompt = build_prompt_instruction_based(p, topic=topic)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="instruction_based",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# --- Ejecutar generación ---
df_instruction_based = generate_instruction_based_dataset()

# --- Resumen y guardado ---
quick_summary(df_instruction_based, "INSTRUCTION-BASED")
save_outputs(df_instruction_based, "instruction_based")
df_instruction_based.head()


### **B) Técnicas de prompting multi-paso**

Las técnicas anteriores se basan en una única llamada de generación al modelo, variando el nivel de información, ejemplos o control semántico proporcionado en el prompt. Sin embargo, existen estrategias más avanzadas que descomponen el proceso de generación en múltiples pasos, permitiendo introducir mecanismos explícitos de evaluación y mejora progresiva del contenido generado. Estas estrategias se agrupan dentro de la denominada multi-step generation y permiten refinar la calidad, coherencia y diversidad del dataset sintético de forma iterativa.

Dentro de este grupo se analizan tres técnicas adicionales:
* iterative prompting
* feedback-driven prompting
* automatic prompt engineering.

**8. Iterative prompting**

Con esta técnica se evalúa la capacidad del modelo para mejorar progresivamente un conjunto de datos sintéticos mediante un proceso iterativo de generación y reformulación. A partir de un primer conjunto de reseñas generadas, el modelo utiliza esas salidas como contexto para producir nuevas versiones o ejemplos adicionales, incorporando refinamientos graduales en términos de estilo, detalle o cobertura temática.

A diferencia de las técnicas de generación directa (zero-shot, one-shot o few-shot), donde cada reseña se produce de forma independiente, en el iterative prompting las generaciones posteriores están condicionadas por los textos ya producidos. El foco ya no está únicamente en la calidad de una reseña aislada, sino en la evolución progresiva del dataset en su conjunto, permitiendo ampliar o ajustar la distribución de ejemplos a lo largo de múltiples iteraciones.

En este enfoque no es necesario disponer de una función de evaluación explícita: la mejora se produce a través de instrucciones de reformulación, expansión o diversificación aplicadas sobre las salidas previas.

¿Hasta qué punto un LLM es capaz de utilizar sus propias salidas como contexto para generar versiones progresivamente más diversas y coherentes de un conjunto de reseñas a lo largo de varias iteraciones?

¿Qué bloques usa?
* Instrucción base
* Salidas generadas en iteraciones anteriores
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito.
* Restricciones globales

**9. Feedback-driven prompting**

Con esta técnica se evalúa la capacidad del modelo para mejorar un conjunto de datos sintéticos utilizando señales explícitas de retroalimentación sobre la calidad de las salidas generadas. Tras producir un conjunto inicial de reseñas, estas son evaluadas mediante criterios externos —por ejemplo, reglas, métricas automáticas, anotaciones humanas o juicios de otro modelo— y esa información se utiliza para filtrar, corregir o regenerar ejemplos.

A diferencia del iterative prompting, donde las mejoras se basan únicamente en la reutilización de las salidas previas como contexto, en el feedback-driven prompting existe una función de evaluación explícita que determina qué ejemplos son aceptables y cuáles deben ser modificados o reemplazados. De este modo, la generación se orienta sistemáticamente hacia una mayor calidad, coherencia o adecuación a los requisitos definidos.

¿Hasta qué punto un LLM es capaz de utilizar señales de evaluación externas para corregir, filtrar y regenerar reseñas, y cómo influye este proceso en la fiabilidad y consistencia del dataset sintético resultante?

¿Qué bloques usa?
* Instrucción base
* Métricas de validación
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito.
* Restricciones globales

**10. Automatic Prompt Engineering**

Con esta técnica se evalúa la capacidad de los modelos para optimizar automáticamente las instrucciones (prompts) utilizadas para generar datos sintéticos. En lugar de fijar un único prompt manualmente, el sistema genera múltiples variantes de la instrucción, las evalúa según una métrica de calidad aplicada a los textos producidos y selecciona o refina aquellas que conducen a mejores resultados.

A diferencia del iterative y del feedback-driven prompting, donde el proceso de optimización actúa principalmente sobre los ejemplos generados, en el Automatic Prompt Engineering el objeto principal de optimización es el propio prompt. El proceso se formula como una búsqueda en el espacio de instrucciones: cada prompt se ejecuta para producir un conjunto de reseñas, estas se puntúan según un criterio de calidad y las mejores instrucciones se reutilizan o modifican en iteraciones posteriores.

Este enfoque permite automatizar el diseño de prompts efectivos que, una vez seleccionados, pueden emplearse dentro de pipelines de generación directa, iterativa o con feedback.

¿Hasta qué punto un LLM es capaz de descubrir y refinar automáticamente prompts que maximicen la calidad, coherencia y alineación de los datos sintéticos generados?

¿Qué bloques usa?
* Instrucción base (dinámica, susceptible de modificación)
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito
* Restricciones globales
* Evaluación automática de los outputs
* Optimización iterativa del prompt

#### Configuración común

In [ ]:
MIN_WORDS = 90
MAX_WORDS = 130

# Dataset viable computacionalmente
N_CONDITIONS = 10
K_PER_CONDITION = 5

# Parámetros de generación
GEN_DEFAULT = dict(
    max_tokens=180,
    temperature=0.8,
    top_p=0.9,
)

In [ ]:
# funciones multi-step

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

def passes_basic_constraints(text: str) -> Tuple[bool, Dict[str, Any]]:
    wc = word_count(text)
    one_paragraph = ("\n" not in text.strip())
    no_lists = not bool(re.search(r"(^|\n)\s*([-*]|\d+\.)\s+", text))
    no_ai_mentions = not bool(re.search(r"\b(IA|inteligencia artificial|LLM|modelo de lenguaje|chatgpt)\b", text, flags=re.IGNORECASE))
    has_detail = bool(re.search(r"\b(porque|por ejemplo|aunque|sin embargo|además)\b", text, flags=re.IGNORECASE))

    ok = (
        MIN_WORDS <= wc <= MAX_WORDS and
        one_paragraph and
        no_lists and
        no_ai_mentions and
        has_detail
    )

    info = {
        "word_count": wc,
        "one_paragraph": one_paragraph,
        "no_lists": no_lists,
        "no_ai_mentions": no_ai_mentions,
        "has_detail_markers": has_detail,
        "ok": ok
    }
    return ok, info

def eval_score(text: str) -> Tuple[float, Dict[str, Any]]:
    """
    Score simple [0,1] a partir de checks. Para feedback-driven y APE.
    """
    ok, info = passes_basic_constraints(text)

    # Ponderación simple: cada check suma.
    checks = [
        MIN_WORDS <= info["word_count"] <= MAX_WORDS,
        info["one_paragraph"],
        info["no_lists"],
        info["no_ai_mentions"],
        info["has_detail_markers"],
    ]
    score = sum(checks) / len(checks)
    info["score"] = score
    return score, info

def make_topic_block(topic: Optional[str]) -> str:
    if not topic:
        return ""
    return f"Tema principal a enfatizar: {topic}\n"

def build_prompt(p: Dict[str, Any], topic: Optional[str] = None, instruction: Optional[str] = None) -> str:
    instr = instruction or base_instruction()
    return "\n".join([
        instr,
        instance_block(p),
        make_topic_block(topic),
        constraints_block()
    ]).strip()

#### Generación...

##### Iterative prompting

In [ ]:
# Tenemos un tope de 512 tokens de entrada, por lo que tenemos que reducir al máximo las instrucciones manteniendo su significado para poder usar esta técnica.

def constraints_block_short() -> str:
    return (
        f"Restricciones: {MIN_WORDS}-{MAX_WORDS} palabras; 1 párrafo (sin listas); "
        "incluye detalles concretos de uso; estilo natural y humano de un cliente real; no menciones IA/LLM/sistemas automáticos; "
        "devuelve solo el texto de la reseña"
    )

def instance_block_compact(p: Dict[str, Any], topic: Optional[str] = None) -> str:
    base = f'Producto="{p["product"]}", Persona="{p["persona"]}", Rating={p["rating"]}/5, Tono="{p["tone"]}"'
    if topic:
        base += f', Tema="{topic}"'
    return base

def build_prompt_iterative_base(p: Dict[str, Any], topic: str) -> str:
    # Iteración 0: generación inicial
    return "\n".join([
        base_instruction(),
        f"Condiciones: {instance_block_compact(p, topic=topic)}",
        constraints_block_short()
    ]).strip()

def build_iterative_rewrite_prompt_compact(p: Dict[str, Any], draft_text: str, topic: Optional[str] = None) -> str:
    # Iteración i: reescritura DIRECTA con mejoras + reseña anterior (completa)
    # Indica las mejoras en una sola línea para ahorrar tokens.
    return "\n".join([
        "Reescribe la reseña mejorando el nivel de detalle sobre la experiencia de uso, evitando repeticiones y manteniendo coherencia con la valoración y el tono.",
        f"Condiciones: {instance_block_compact(p, topic=topic)}",
        constraints_block_short(),
        "Original:",
        draft_text.strip(),
        "Devuelve solo la reseña final."
    ]).strip()

def generate_iterative_prompting_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    n_iters: int = 1,
    seed: int = 123,
    topics: List[str] = REVIEW_TOPICS,
) -> pd.DataFrame:
    rng = random.Random(seed)
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ITERATIVE PROMPTING"):
        for _ in range(k_per_condition):
            # Randomized
            topic = rng.choice(topics)

            # Iteración 0: reseña base
            prompt_base = build_prompt_iterative_base(p, topic=topic)
            text = chat_generate(prompt_base, gen_params)
            attempts = 1
            last_prompt = prompt_base

            # Iteraciones de refinamiento
            for _it in range(n_iters):
                prompt_rewrite = build_iterative_rewrite_prompt_compact(p, text, topic=topic)
                text = chat_generate(prompt_rewrite, gen_params)
                attempts += 1
                last_prompt = prompt_rewrite

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="iterative_prompting",
                    p=p,
                    prompt=last_prompt,
                    text=text,
                    topic=topic,
                    attempts=attempts
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# Ejecutar generación
df_iterative_prompting = generate_iterative_prompting_dataset(
    n_iters=1
)

# Resumen y guardado
quick_summary(df_iterative_prompting, "ITERATIVE PROMPTING")
save_outputs(df_iterative_prompting, "iterative_prompting")
df_iterative_prompting.head()

##### Feedback-driven prompting

In [ ]:
def build_feedback_fix_prompt(
    p: Dict[str, Any],
    bad_text: str,
    eval_info: Dict[str, Any],
    topic: Optional[str] = None
) -> str:
    issues = []
    if not (MIN_WORDS <= eval_info["word_count"] <= MAX_WORDS):
        issues.append(f"- Longitud incorrecta: {eval_info['word_count']} palabras (objetivo {MIN_WORDS}-{MAX_WORDS}).")
    if not eval_info["one_paragraph"]:
        issues.append("- Debe ser un único párrafo (sin saltos de línea).")
    if not eval_info["no_lists"]:
        issues.append("- Prohibido usar listas o numeración.")
    if not eval_info["no_ai_mentions"]:
        issues.append("- Prohibido mencionar IA, modelos de lenguaje o sistemas automáticos.")
    if not eval_info["has_detail_markers"]:
        issues.append("- Falta detalle/conectores; añade matices y experiencia concreta (porque, por ejemplo, aunque...).")

    issues_txt = "\n".join(issues) if issues else "- Ninguno."

    return "\n".join([
        "Vas a corregir una reseña para que cumpla estrictamente requisitos.",
        "",
        instance_block(p).strip(),
        "",
        make_topic_block(topic).strip(),
        constraints_block().strip(),
        "",
        "Evaluación externa: la reseña NO cumple por estos motivos:",
        issues_txt,
        "",
        "Reseña a corregir:",
        bad_text.strip(),
        "",
        "Salida: devuelve únicamente el texto final corregido."
    ]).strip()

def generate_feedback_driven_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    max_rounds: int = 4,
    use_topics: bool = True,
    seed: int = 123
) -> pd.DataFrame:
    random.seed(seed)
    rows = []
    sample_id = 0

    conditions = conditions[:N_CONDITIONS]

    for p in tqdm(conditions, desc="Generando FEEDBACK-DRIVEN"):
        for _ in range(k_per_condition):
            topic = random.choice(REVIEW_TOPICS) if use_topics else None

            prompt_base = build_prompt(p, topic=topic)
            text = chat_generate(prompt_base, gen_params)
            attempts = 1

            score, info = eval_score(text)
            rounds = 0
            last_prompt = prompt_base
            eval_history = []

            while (not info["ok"]) and rounds < max_rounds:
                eval_history.append(json.dumps(info, ensure_ascii=False))

                last_prompt = build_feedback_fix_prompt(p, text, info, topic=topic)
                text = chat_generate(last_prompt, {**gen_params, "temperature": 0.6})
                attempts += 1

                score, info = eval_score(text)
                rounds += 1

            row = make_row(
                sample_id=sample_id,
                technique="feedback_driven_prompting",
                p=p,
                prompt=last_prompt,
                text=text,
                topic=topic,
                attempts=attempts
            )
            row.update({
                "score": score,
                "ok": info.get("ok"),
                "word_count": info.get("word_count"),
                "rounds": rounds,
                "eval_history": "\n---\n".join(eval_history)
            })

            rows.append(row)
            sample_id += 1

    return pd.DataFrame(rows)

# Ejecutar generación
df_feedback = generate_feedback_driven_dataset()

# Resumen y guardado
quick_summary(df_feedback, "FEEDBACK-DRIVEN")
save_outputs(df_feedback, "feedback_driven_prompting")
df_feedback.head()

##### Automatic prompt engineering

In [ ]:
def build_ape_propose_prompt(task_desc: str, m: int = 8) -> str:
    return f"""Genera {m} variantes de una instrucción en español para un LLM.
Objetivo: {task_desc}
Requisitos:
- Debe ser una sola frase de instrucción (no incluyas restricciones globales).
- Debe pedir una reseña realista de un cliente.
- No menciones IA/LLMs.
Devuelve SOLO una lista numerada del 1 al {m}, una instrucción por línea.
""".strip()

def parse_numbered_list(text: str, m: int) -> List[str]:
    lines = []
    for l in text.splitlines():
        if re.search(r"^\s*\d+\.", l):
            lines.append(re.sub(r"^\s*\d+\.\s*", "", l).strip())
    lines = [l for l in lines if l]
    return lines[:m] if lines else []

def generate_ape_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    m_candidates: int = 8,
    n_trials_per_candidate: int = 1,
    use_topics: bool = True,
    seed: int = 123
) -> pd.DataFrame:
    random.seed(seed)
    rows = []
    sample_id = 0

    conditions = conditions[:N_CONDITIONS]
    task_desc = "Escribir reseñas de productos de negocio local, en un solo párrafo, naturales y con detalles de experiencia."

    for p in tqdm(conditions, desc="Generando APE"):
        for _ in range(k_per_condition):
            topic = random.choice(REVIEW_TOPICS) if use_topics else None

            # 1) Proponer candidatos de instrucción
            proposal_prompt = build_ape_propose_prompt(task_desc, m=m_candidates)
            proposal_out = chat_generate(proposal_prompt, {**gen_params, "max_tokens": 240, "temperature": 0.9, "top_p": 0.95})
            candidates = parse_numbered_list(proposal_out, m_candidates) or [base_instruction()]
            attempts = 1

            # 2) Evaluar candidatos (ejecución + score reglas)
            cand_audit = []
            best_instr = None
            best_avg = -1.0

            for instr in candidates:
                scores = []
                for _t in range(n_trials_per_candidate):
                    # Aquí reutilizamos tu build_prompt con instruction explícita
                    prompt_try = build_prompt(p, topic=topic, instruction=instr)
                    text_try = chat_generate(prompt_try, gen_params)
                    attempts += 1
                    s, _info = eval_score(text_try)
                    scores.append(s)

                avg = sum(scores) / len(scores)
                cand_audit.append({"instruction": instr, "avg_score": avg, "scores": scores})

                if avg > best_avg:
                    best_avg = avg
                    best_instr = instr

            best_instr = best_instr or base_instruction()

            # 3) Generar muestra final con el mejor prompt
            final_prompt = build_prompt(p, topic=topic, instruction=best_instr)
            final_text = chat_generate(final_prompt, gen_params)
            attempts += 1

            final_score, final_info = eval_score(final_text)

            row = make_row(
                sample_id=sample_id,
                technique="automatic_prompt_engineering",
                p=p,
                prompt=final_prompt,
                text=final_text,
                topic=topic,
                attempts=attempts
            )
            row.update({
                "score": final_score,
                "ok": final_info.get("ok"),
                "word_count": final_info.get("word_count"),
                "best_instruction": best_instr,
                "best_instruction_avg_score": best_avg,
                "candidate_audit_json": json.dumps(cand_audit, ensure_ascii=False),
                "proposal_prompt": proposal_prompt
            })

            rows.append(row)
            sample_id += 1

    return pd.DataFrame(rows)

# Ejecutar generación
df_ape = generate_ape_dataset()

# Resumen y guardado
quick_summary(df_ape, "APE")
save_outputs(df_ape, "automatic_prompt_engineering")
df_ape.head()

## Unión del datatset

In [ ]:
import pandas as pd
from pathlib import Path

# Carpeta raíz de Colab
DATA_DIR = Path("/content")

# Columnas que quieres conservar
keep_cols = [
    "technique",
    "condition_id",
    "product",
    "persona",
    "rating",
    "tone",
    "topic",
    "prompt",
    "text",
    "attempts"
]

# Leer excels
excel_files = [
    f for f in DATA_DIR.glob("*.xlsx")
    if not f.name.startswith("~$")
]

print("Excels detectados:")
for f in excel_files:
    print("-", f.name)

dfs = []

for f in excel_files:
    df = pd.read_excel(f)

    # Ver columnas disponibles
    available = set(df.columns)
    missing = [c for c in keep_cols if c not in available]

    if missing:
        print(f"{f.name} NO tiene columnas: {missing}")

    # Quedarse solo con las que existan de keep_cols
    cols_present = [c for c in keep_cols if c in df.columns]
    df = df[cols_present]

    dfs.append(df)

# Concatenar todo
df_all = pd.concat(dfs, ignore_index=True)

# Reordenar columnas
df_all = df_all.reindex(columns=keep_cols)

# Guardar Excel unificado
output_file = "/content/dataset.xlsx"
df_all.to_excel(output_file, index=False)

print(f"Dataset unificado creado en: {output_file}")
print(f"Total de filas: {len(df_all)}")
print("Columnas finales:", list(df_all.columns))
